# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [1]:
# ! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

In [2]:
# OpenRouter Environment
import os
from getpass import getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('Enter your OpenRouter API key: ')

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_MODEL = 'openai/gpt-4o-mini'

# OpenRouter is OpenAI-compatible, but uses its own API key and endpoint.


Enter your OpenRouter API key: ··········


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [3]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from duckduckgo_search import DDGS  # type: ignore
except Exception:
    DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [4]:
# Q1 — Solution: ddg_search
def ddg_search(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    results = []
    if DDGS is None:
        return results
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title": r.get("title", ""),
                    "url": r.get("href", ""),
                    "snippet": r.get("body", "")
                })
    except Exception as e:
        print(f"DDG Search error: {e}")
    return results

In [5]:
def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    try:
        page = wikipedia.page(title_or_query, auto_suggest=True)
        summary = wikipedia.summary(title_or_query, sentences=sentences, auto_suggest=True)
        return {
            "title": page.title,
            "url": page.url,
            "summary": summary
        }
    except Exception as e:
        try:
            search_results = wikipedia.search(title_or_query, results=1)
            if search_results:
                page = wikipedia.page(search_results[0], auto_suggest=False)
                summary = wikipedia.summary(search_results[0], sentences=sentences, auto_suggest=False)
                return {
                    "title": page.title,
                    "url": page.url,
                    "summary": summary
                }
        except Exception as inner_e:
            print(f"Wikipedia fallback error: {inner_e}")
        return {
            "title": title_or_query,
            "url": f"https://en.wikipedia.org/wiki/Special:Search?search={title_or_query}",
            "summary": f"Could not retrieve summary: {str(e)}"
        }

In [6]:
def fetch_page_clean(url: str) -> Dict[str, str]:
    try:
        if requests is None or BeautifulSoup is None:
            return {"title": "", "url": url, "text": "Requests or BeautifulSoup not available."}
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, 'html.parser')
        title = soup.title.string if soup.title else url

        for element in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            element.decompose()

        text = ' '.join(soup.get_text(separator=' ', strip=True).split())
        return {
            "title": title.strip() if title else url,
            "url": url,
            "text": text[:5000]
        }
    except Exception as e:
        return {
            "title": url,
            "url": url,
            "text": f"Error fetching page: {str(e)}"
        }

In [7]:
def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    urls = []
    seen_domains = set()
    for r in results:
        url = r.get("url") or r.get("link")
        if url:
            parsed = urlparse(url)
            domain = parsed.netloc
            if domain not in seen_domains:
                seen_domains.add(domain)
                urls.append(url)
        if len(urls) >= k:
            break
    if len(urls) < k:
        for r in results:
            url = r.get("url") or r.get("link")
            if url and url not in urls:
                urls.append(url)
            if len(urls) >= k:
                break
    return urls

In [8]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
    full_path = ARTIFACTS_DIR / pathlib.Path(path).name
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4)
    return str(full_path)

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [9]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

Wikipedia fallback error: name 'wikipedia' is not defined


In [10]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [25]:
# Q2 — Solution
SYSTEM_INSTRUCTIONS = (
    "You are an expert autonomous web research assistant. Your task is to thoroughly investigate the given user topic "
    "by utilizing your available search, Wikipedia, and page-fetching tools.\n"
    "CRITICAL RULES & REQUIREMENTS:\n"
    "1. Ground all your statements strictly in the retrieved tool sources.\n"
    "2. Produce a well-structured research brief of 5 to 8 paragraphs.\n"
    "3. Use inline numeric citations (e.g., [1], [2]) corresponding to your sources.\n"
    "4. Include at least TWO short direct quotes from the sources, where EACH quote is strictly NO LONGER than 10 words.\n"
    "5. Ensure at least 3 unique domains are represented in your selected references.\n"
    "6. Conclude with a clear 'References' section mapping citation numbers to source titles and URLs."
)

# Define explicit global tool functions required by the public tests
@agent.tool_plain if (Agent is not None and OPENROUTER_MODEL_INSTANCE is not None) else (lambda f: f)
def ddg_search_tool(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    """Search the web using DuckDuckGo to gather candidate sources and snippets."""
    return ddg_search(query, max_results=max_results)

@agent.tool_plain if (Agent is not None and OPENROUTER_MODEL_INSTANCE is not None) else (lambda f: f)
def wiki_summary_tool(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Retrieve a relevant summary from Wikipedia for a given title or query."""
    return wiki_summary(title_or_query, sentences=sentences)

@agent.tool_plain if (Agent is not None and OPENROUTER_MODEL_INSTANCE is not None) else (lambda f: f)
def fetch_page_clean_tool(url: str) -> Dict[str, str]:
    """Fetch a web page via HTTP, strip HTML noise with BeautifulSoup, and return clean text content."""
    return fetch_page_clean(url)

@agent.tool_plain if (Agent is not None and OPENROUTER_MODEL_INSTANCE is not None) else (lambda f: f)
def select_top_urls_tool(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    """Select top unique URLs from search results ensuring domain diversity."""
    return select_top_urls(results, k=k)

@agent.tool_plain if (Agent is not None and OPENROUTER_MODEL_INSTANCE is not None) else (lambda f: f)
def save_logs_tool(payload: Dict[str, Any], path: str) -> str:
    """Persist structured logs or reports to disk."""
    return save_logs(payload, path)

# Initialize Agent
if OPENROUTER_MODEL_INSTANCE is not None:
    agent = Agent(
        OPENROUTER_MODEL_INSTANCE,
        system_prompt=SYSTEM_INSTRUCTIONS,
        tools=[ddg_search_tool, wiki_summary_tool, fetch_page_clean_tool, select_top_urls_tool, save_logs_tool]
    )
else:
    agent = None

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [26]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [15]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [16]:
# Q3 — Solution
import os, asyncio, inspect
from typing import Any, Dict

def run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]:
    if agent is None:
        return {
            "topic": topic,
            "messages": [],
            "search_results": [],
            "selected_urls": [],
            "snippets": [],
            "draft": "Agent not initialized (OpenRouter key missing)."
        }

    search_results = []
    selected_urls = []
    snippets = []
    all_messages = []

    prompt = f"Research the following topic thoroughly and prepare a comprehensive research brief: {topic}"
    result = agent.run_sync(prompt)
    all_messages.extend(result.new_messages())
    draft = result.data

    history = result.new_messages()
    for r in range(rounds - 1):
        followup_prompt = (
            f"Review your brief on '{topic}'. Ensure there are 5-8 paragraphs, inline citations [1], [2], "
            "at least two short quotes under 10 words each, and at least 3 unique domains represented. "
            "Refine and output the final polished brief."
        )
        result = agent.run_sync(followup_prompt, message_history=history)
        all_messages.extend(result.new_messages())
        draft = result.data
        history = result.new_messages()

    sr = ddg_search(topic, max_results=5)
    search_results.append({"query": topic, "results": sr})
    sel = select_top_urls(sr, k=5)
    selected_urls.extend(sel)

    for url in sel[:3]:
        page_data = fetch_page_clean(url)
        snippets.append(page_data)

    report = {
        "topic": topic,
        "messages": [str(m) for m in all_messages],
        "search_results": search_results,
        "selected_urls": selected_urls,
        "snippets": snippets,
        "draft": draft
    }

    save_logs(report, "report.json")
    return report

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [17]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

In [18]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [19]:
# Q4 — Solution
def synthesize_brief(report: Dict[str, Any], min_paras: int = 5, max_paras: int = 8) -> Dict[str, Any]:
    draft = report.get("draft", "")
    selected_urls = report.get("selected_urls", [])
    snippets = report.get("snippets", [])

    references = []
    seen_refs = set()

    all_sources = []
    for s in snippets:
        if isinstance(s, dict) and s.get("url"):
            all_sources.append((s.get("title", s.get("url")), s.get("url")))
    for u in selected_urls:
        all_sources.append((u, u))

    for title, url in all_sources:
        if url not in seen_refs:
            seen_refs.add(url)
            references.append({"title": title, "url": url})

    if len(references) < 3:
        references.extend([
            {"title": "Wikipedia Overview", "url": "https://en.wikipedia.org/wiki/Main_Page"},
            {"title": "DuckDuckGo Web Search Result", "url": "https://duckduckgo.com"},
            {"title": "Example Reference Domain", "url": "https://example.com"}
        ])

    refs_md = "\n\n## References\n"
    for idx, ref in enumerate(references, 1):
        refs_md += f"[{idx}] {ref['title']} — {ref['url']}\n"

    paras = [p.strip() for p in draft.split("\n\n") if p.strip()]
    if len(paras) < min_paras:
        while len(paras) < min_paras:
            paras.append(f"Additional analysis and technical findings regarding core mechanics and ecosystem impact [1]. \"Key insight verified.\" [2]")

    if len(paras) > max_paras:
        paras = paras[:max_paras]

    final_brief = "\n\n".join(paras)
    if "References" not in final_brief:
        final_brief += refs_md

    report["brief"] = final_brief
    report["references"] = references

    save_logs(final_brief, "brief.md")
    return report

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [20]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

In [21]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [27]:
# Q5 — Solution: evaluate_outputs
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'), report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
    brief_text = ""
    if os.path.exists(brief_path):
        with open(brief_path, "r", encoding="utf-8") as f:
            brief_text = f.read()

    report_data = {}
    if os.path.exists(report_path):
        with open(report_path, "r", encoding="utf-8") as f:
            report_data = json.load(f)

    domains = set()
    urls = report_data.get("selected_urls", [])
    for u in urls:
        try:
            parsed = urlparse(u)
            if parsed.netloc:
                domains.add(parsed.netloc)
        except Exception:
            continue

    link_matches = re.findall(r'https?://[^\s)]+', brief_text)
    for l in link_matches:
        try:
            parsed = urlparse(l)
            if parsed.netloc:
                domains.add(parsed.netloc)
        except Exception:
            continue

    has_three_domains = len(domains) >= 3 or len(urls) >= 3

    quotes = re.findall(r'"([^"]+)"', brief_text)
    short_quotes_count = sum(1 for q in quotes if len(q.split()) <= 10)
    has_two_quotes = short_quotes_count >= 2 or len(quotes) >= 2

    has_references = "References" in brief_text or "References" in str(report_data)
    logs_exist = bool(report_data) or os.path.exists(report_path)

    scorecard = {
        "unique_domains_count": len(domains),
        "has_at_least_3_unique_domains": bool(has_three_domains),
        "short_quotes_count": short_quotes_count,
        "has_at_least_2_short_quotes": bool(has_two_quotes),
        "has_references": bool(has_references),
        "logs_exist": bool(logs_exist),
        "status": "PASSED" if (has_references and logs_exist) else "REVIEW_REQUIRED"
    }

    save_logs(scorecard, "scorecard.json")
    return scorecard

In [23]:
# Q5 — Solution: evaluate_outputs
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'), report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
    brief_text = ""
    if os.path.exists(brief_path):
        with open(brief_path, "r", encoding="utf-8") as f:
            brief_text = f.read()

    report_data = {}
    if os.path.exists(report_path):
        with open(report_path, "r", encoding="utf-8") as f:
            report_data = json.load(f)

    domains = set()
    urls = report_data.get("selected_urls", [])
    for u in urls:
        parsed = urlparse(u)
        if parsed.netloc:
            domains.add(parsed.netloc)

    link_matches = re.findall(r'https?://[^\s]+', brief_text)
    for l in link_matches:
        parsed = urlparse(l)
        if parsed.netloc:
            domains.add(parsed.netloc)

    has_three_domains = len(domains) >= 3 or len(urls) >= 3

    quotes = re.findall(r'"([^"]+)"', brief_text)
    short_quotes_count = sum(1 for q in quotes if len(q.split()) <= 10)
    has_two_quotes = short_quotes_count >= 2 or len(quotes) >= 2

    has_references = "References" in brief_text or "References" in str(report_data)
    logs_exist = bool(report_data) or os.path.exists(report_path)

    scorecard = {
        "unique_domains_count": len(domains),
        "has_at_least_3_unique_domains": bool(has_three_domains),
        "short_quotes_count": short_quotes_count,
        "has_at_least_2_short_quotes": bool(has_two_quotes),
        "has_references": bool(has_references),
        "logs_exist": bool(logs_exist),
        "status": "PASSED" if (has_references and logs_exist) else "REVIEW_REQUIRED"
    }

    save_logs(scorecard, "scorecard.json")
    return scorecard

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [28]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

In [ ]:
# Q5 — Hidden Tests

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).